# Neumann Series Sparsification — Interactive Exploration

Modular pipeline for experimenting with the Neumann sparsification method.
Each stage (transform, scoring, rescaling) is a pluggable function you can swap.

**Sections:**
1. Setup & imports
2. Run a single config (interactive)
3. Compare transform variants
4. Load & visualize grid search results (56 configs)
5. Scatter & heatmap analysis

In [ ]:
import sys, os, json, time
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(repo_root, 'src', 'python'))

from graph_sparsification.generators import configuration_model
from graph_sparsification.sparsifiers import (
    metric_backbone, metric_backbone_rescaled,
    effective_resistance_sparsify, to_proximity,
)
from graph_sparsification.sir import sir_monte_carlo, calibrate_beta
from graph_sparsification.neumann_sparsifier import (
    neumann_sparsify,
    # Transforms
    transform_proximity, transform_exp,
    # Scoring
    score_adaptive, score_full_importance, score_structural_x_prox, score_structural_only,
    # Rescaling
    rescale_adaptive, rescale_global_only, rescale_sinkhorn_only, rescale_global_plus_sinkhorn,
    # Core
    compute_importance, build_symmetric_prox, global_rescale, sinkhorn_rescale,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

def _mse(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.mean((a[m] - b[m])**2))

print('Ready.')

## 2. Run a single config

Change the degree/weight functions and parameters below to explore any config interactively.

In [ ]:
# ── Configuration (edit these) ──────────────────────────────────────
N_NODES = 500
N_SIR = 200
SEED = 42

deg_fn = lambda n, rng: rng.integers(1, 51, size=n)          # Unif(1,50)
wt_fn  = lambda m, rng: rng.lognormal(2.0, 1.0, size=m)      # LogN(2,1)

# ── Pipeline options (swap these to experiment) ────────────────────
TRANSFORM = transform_proximity   # or transform_exp
SCORING   = score_adaptive         # or score_full_importance, score_structural_x_prox
RESCALING = rescale_adaptive       # or rescale_global_only, rescale_sinkhorn_only
N_PERMS   = 50

# ── Generate & sparsify ───────────────────────────────────────────
rng = np.random.default_rng(SEED)
W_dist = configuration_model(N_NODES, deg_fn, wt_fn, rng=rng)
W_prox = to_proximity(W_dist)
n_edges = sparse.triu(W_dist).nnz

beta, _ = calibrate_beta(W_prox, gamma=1.0, n_calibration_runs=30, rng=rng, verbose=False)

W_mbb_dist = metric_backbone(W_dist)
n_mbb = sparse.triu(W_mbb_dist).nnz
retention = n_mbb / n_edges

W_mbbr = metric_backbone_rescaled(W_dist)
W_effr = effective_resistance_sparsify(W_prox, n_edges=n_mbb, rng=np.random.default_rng(SEED))

t0 = time.time()
W_neur = neumann_sparsify(
    W_dist, n_mbb,
    transform_fn=TRANSFORM, score_fn=SCORING, rescale_fn=RESCALING,
    n_perms=N_PERMS, seed=SEED, verbose=True,
)
t_neur = time.time() - t0

print(f'\nGraph: {N_NODES} nodes, {n_edges} edges')
print(f'MBB: {n_mbb} edges ({100*retention:.1f}% retention)')
print(f'beta={beta:.4f}, Neumann time={t_neur:.1f}s')

In [ ]:
# ── SIR evaluation ─────────────────────────────────────────────────
initial = [int(np.argmax(np.array(W_prox.sum(axis=1)).ravel()))]
sir_kw = dict(n_runs=N_SIR, rng=np.random.default_rng(100))

p_orig = sir_monte_carlo(W_prox, beta, 1.0, initial, **sir_kw)['infection_prob']
p_mbbr = sir_monte_carlo(W_mbbr, beta, 1.0, initial, **sir_kw)['infection_prob']
p_effr = sir_monte_carlo(W_effr, beta, 1.0, initial, **sir_kw)['infection_prob']
p_neur = sir_monte_carlo(W_neur, beta, 1.0, initial, **sir_kw)['infection_prob']

mse_mbbr = _mse(p_orig, p_mbbr)
mse_effr = _mse(p_orig, p_effr)
mse_neur = _mse(p_orig, p_neur)

best = min(mse_mbbr, mse_effr)
status = 'WIN' if mse_neur <= best else f'{mse_neur/best:.2f}x'

print(f'MSE  MBBr={mse_mbbr:.6f}  EffR={mse_effr:.6f}  Neumann={mse_neur:.6f}  [{status}]')

# ── Plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, p_sp, name, mse_val in [
    (axes[0], p_mbbr, f'MBBr (MSE={mse_mbbr:.5f})', mse_mbbr),
    (axes[1], p_effr, f'EffR (MSE={mse_effr:.5f})', mse_effr),
    (axes[2], p_neur, f'Neumann (MSE={mse_neur:.5f})', mse_neur),
]:
    ax.scatter(p_orig, p_sp, alpha=0.3, s=10, edgecolors='none')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('Original P(infected)')
    ax.set_ylabel('Sparsified P(infected)')
    ax.set_title(name)
    ax.set_xlim(0, max(p_orig.max(), 0.01) * 1.1)
    ax.set_ylim(0, max(p_orig.max(), 0.01) * 1.1)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 3. Compare transform variants

Test proximity-based vs exp-based Neumann weights on the same graph.

In [ ]:
# Re-use the graph from section 2; compare transforms
transforms = {
    'proximity (default)': transform_proximity,
    'exp(-d/tau)':         transform_exp,
}

print(f"{'Transform':<22} {'MSE':>10} {'vs MBBr':>8} {'vs EffR':>8}")
print('-' * 55)

for name, tfn in transforms.items():
    W_t = neumann_sparsify(
        W_dist, n_mbb,
        transform_fn=tfn, score_fn=SCORING, rescale_fn=RESCALING,
        n_perms=N_PERMS, seed=SEED, verbose=False,
    )
    p_t = sir_monte_carlo(W_t, beta, 1.0, initial, **sir_kw)['infection_prob']
    mse_t = _mse(p_orig, p_t)
    print(f"{name:<22} {mse_t:>10.6f} {mse_t/mse_mbbr:>7.2f}x {mse_t/mse_effr:>7.2f}x")

## 4. Load & visualize grid search results (56 configs)

Load precomputed results from `grid_search_results.json` (ran with the old exp-based transform).
These show the baseline performance across all 56 degree x weight combinations.

In [ ]:
with open('grid_search_results.json') as f:
    all_results = json.load(f)

print(f'Loaded {len(all_results)} configs')

deg_names = ['Unif(1,50)', 'Unif(1,100)', 'Exp(30)', 'Exp(60)',
             'LogN(3.26,0.66)', 'LogN(3.26,2)', 'Pareto(2.5,20)', 'Pareto(1.5,30)']
wt_names = ['Exp(1/30)', 'Exp(1)', 'Exp(30)', 'LogN(2,1)',
            'LogLogN(1.2,0.4)', 'LogLogN(1.2,0.8)', 'LogLogN(2,0.8)']
n_deg, n_wt = len(deg_names), len(wt_names)

res_lookup = {(r['deg'], r['wt']): r for r in all_results}

def build_grid(key):
    grid = np.full((n_deg, n_wt), np.nan)
    for i, d in enumerate(deg_names):
        for j, w in enumerate(wt_names):
            if (d, w) in res_lookup:
                grid[i, j] = res_lookup[(d, w)][key]
    return grid

# Summary stats
mse_neur = np.array([r['mse_neur'] for r in all_results])
mse_mbbr = np.array([r['mse_mbbr'] for r in all_results])
mse_effr = np.array([r['mse_effr'] for r in all_results])
best = np.minimum(mse_mbbr, mse_effr)

n_wins = np.sum(mse_neur <= best)
n_beat_effr = np.sum(mse_neur <= mse_effr)
n_beat_mbbr = np.sum(mse_neur <= mse_mbbr)
n = len(all_results)

print(f'Wins vs best: {n_wins}/{n}  |  Beats EffR: {n_beat_effr}/{n}  |  Beats MBBr: {n_beat_mbbr}/{n}')

## 5. Heatmaps & scatter analysis

In [ ]:
def plot_heatmap(grid, title, cmap='viridis', fmt='.4f', vmin=None, vmax=None,
                 annot_color=None):
    fig, ax = plt.subplots(figsize=(max(9, n_wt * 1.2), max(5, n_deg * 0.75)))
    im = ax.imshow(grid, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(n_wt)); ax.set_xticklabels(wt_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(n_deg)); ax.set_yticklabels(deg_names, fontsize=9)
    ax.set_xlabel('Weight distribution'); ax.set_ylabel('Degree distribution')
    ax.set_title(title, fontsize=13, pad=10)
    for i in range(n_deg):
        for j in range(n_wt):
            v = grid[i, j]
            if np.isfinite(v):
                c = annot_color or ('white' if im.norm(v) < 0.5 else 'black')
                ax.text(j, i, f'{v:{fmt}}', ha='center', va='center', fontsize=7, color=c)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    return fig

# MSE difference: Neumann - MBBr (blue = Neumann better)
grid_diff = build_grid('mse_neur') - build_grid('mse_mbbr')
vabs = np.nanpercentile(np.abs(grid_diff), 95)
plot_heatmap(grid_diff, 'MSE difference (Neumann - MBBr)\nblue = Neumann better',
             cmap='RdBu', fmt='.4f', vmin=-vabs, vmax=vabs, annot_color='black')
plt.show()

# Win/loss grid
grid_status = np.full((n_deg, n_wt), np.nan)
for i, d in enumerate(deg_names):
    for j, w in enumerate(wt_names):
        if (d, w) in res_lookup:
            r = res_lookup[(d, w)]
            b = min(r['mse_mbbr'], r['mse_effr'])
            grid_status[i, j] = 1.0 if r['mse_neur'] <= b else (0.5 if r['mse_neur'] <= b * 1.1 else 0.0)

plot_heatmap(grid_status, 'Neumann vs best baseline\n1.0=WIN, 0.5=within 10%, 0.0=lose',
             cmap='RdYlGn', fmt='.1f', vmin=0, vmax=1, annot_color='black')
plt.show()

In [ ]:
# Scatter: Neumann MSE vs MBBr and EffR
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, baseline, name, color in [
    (axes[0], mse_mbbr, 'MBBr', '#2196F3'),
    (axes[1], mse_effr, 'EffR', '#FF9800'),
]:
    ax.scatter(baseline, mse_neur, alpha=0.6, s=30, c=color, edgecolors='k', linewidth=0.5)
    lim = max(baseline.max(), mse_neur.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='y=x')
    ax.set_xlabel(f'{name} MSE'); ax.set_ylabel('Neumann MSE')
    ax.set_title(f'Neumann vs {name}')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.legend(); ax.grid(alpha=0.3)
    below = np.sum(mse_neur < baseline)
    above = np.sum(mse_neur > baseline)
    ax.text(0.05, 0.92, f'Neumann better: {below}/{n}', transform=ax.transAxes, fontsize=9, color='green')
    ax.text(0.05, 0.85, f'{name} better: {above}/{n}', transform=ax.transAxes, fontsize=9, color='red')

plt.tight_layout()
plt.show()

# Retention vs performance
retentions = np.array([r['retention'] for r in all_results]) * 100
ratios_mbbr = mse_neur / np.maximum(mse_mbbr, 1e-10)
ratios_effr = mse_neur / np.maximum(mse_effr, 1e-10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(retentions, ratios_mbbr, alpha=0.6, s=30, c='#2196F3', edgecolors='k', linewidth=0.5, label='vs MBBr')
ax.scatter(retentions, ratios_effr, alpha=0.6, s=30, c='#FF9800', edgecolors='k', linewidth=0.5, label='vs EffR')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='parity')
ax.set_xlabel('Edge retention (%)'); ax.set_ylabel('MSE ratio (Neumann / baseline)')
ax.set_title('Performance vs edge retention')
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0, min(ratios_mbbr.max(), 10) * 1.1)
plt.tight_layout()
plt.show()